# Flystral Inference Server

Loads the fine-tuned Ministral 3B LoRA adapter from HuggingFace and serves it as an API via ngrok.

**Steps:**
1. Run all cells
2. Copy the ngrok URL printed at the bottom
3. Set `FLYSTRAL_ENDPOINT=<ngrok_url>` in your `.env` file
4. Your server will now use your fine-tuned model for Flystral inference

**Runtime:** Use a T4 GPU (free tier works)

In [2]:
!pip install -q transformers peft accelerate flask pyngrok huggingface_hub torch pillow

## Load model

In [4]:
import torch
from transformers import AutoProcessor, Mistral3ForConditionalGeneration
from peft import PeftModel

BASE_MODEL = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
ADAPTER = "BenBarr/flystral"

print("[1] Loading processor from base model...")
processor = AutoProcessor.from_pretrained(BASE_MODEL)

print("[2] Loading base model...")
model = Mistral3ForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
)

print("[3] Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER)
model = model.merge_and_unload()
model = model.cuda()
model.eval()

print(f"Model loaded on {next(model.parameters()).device}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

[1] Loading processor from base model...
[2] Loading base model...
Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.12s/it]
[3] Loading LoRA adapter...
Model loaded on cuda:0
GPU memory: 6.28 GB


## Quick test — verify inference works before starting server

In [6]:
from PIL import Image

test_img = Image.new("RGB", (320, 320), color="blue")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Output the raw telemetry for this frame."},
        ],
    }
]

text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=text, images=[test_img], return_tensors="pt").to("cuda")

print(f"Input IDs shape: {inputs['input_ids'].shape}")
print(f"Pixel values shape: {inputs['pixel_values'].shape}")

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=100, do_sample=False)

input_len = inputs["input_ids"].shape[-1]
result = processor.decode(output_ids[0][input_len:], skip_special_tokens=True)
print(f"\nModel output: {result}")
print("\nInference OK!")

Input IDs shape: torch.Size([1, 792])
Pixel values shape: torch.Size([1, 1, 3, 512, 512])

Model output: 0.2147, -0.0312, 0.0089, 0.0014, 0.1893, -0.0245, 0.0112, 0.0031, 0.2034, -0.0198, 0.0076, 0.0008, 0.1756, -0.0289, 0.0134, 0.0042, 0.2211, -0.0156, 0.0091, 0.0019

Inference OK!


## Start API server

In [8]:
import base64, io, json, time
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": ADAPTER, "device": str(next(model.parameters()).device)})

@app.route("/predict", methods=["POST"])
def predict():
    t0 = time.time()
    data = request.json
    image_b64 = data.get("image", "")

    if not image_b64:
        return jsonify({"error": "no image provided"}), 400

    img_bytes = base64.b64decode(image_b64)
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Output the raw telemetry for this frame."},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=text, images=[img], return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()

    values = []
    for tok in raw.replace(",", " ").split():
        try:
            values.append(float(tok))
        except ValueError:
            continue

    result = {
        "vx": values[0] if len(values) > 0 else 2.0,
        "vy": values[1] if len(values) > 1 else 0.0,
        "vz": values[2] if len(values) > 2 else 0.0,
        "yaw_rate": values[3] if len(values) > 3 else 0.0,
        "raw": raw,
        "inference_ms": int((time.time() - t0) * 1000),
    }
    return jsonify(result)

print("Flask app ready")

Flask app ready


In [9]:
from pyngrok import ngrok

# Set your ngrok auth token (free at https://ngrok.com)
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

public_url = ngrok.connect(5000)
print("=" * 60)
print(f"FLYSTRAL ENDPOINT: {public_url}")
print(f"\nSet in .env: FLYSTRAL_ENDPOINT={public_url}")
print(f"\nHealth check: {public_url}/health")
print("=" * 60)

app.run(port=5000)

FLYSTRAL ENDPOINT: NgrokTunnel: "https://crestfallenly-untuneable-dorthy.ngrok-free.dev" -> "http://localhost:5000"

Set in .env: FLYSTRAL_ENDPOINT=https://crestfallenly-untuneable-dorthy.ngrok-free.dev

Health check: https://crestfallenly-untuneable-dorthy.ngrok-free.dev/health
 * Serving Flask app '__main__'
 * Debug mode: off
INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [01/Mar/2026 13:30:06] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/Mar/2026 13:30:51] "POST /predict HTTP/1.1" 200 -
